<a href="https://colab.research.google.com/github/SofiaKakou/Customer_Churn_Prediction/blob/main/notebooks/04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.Introduction

This notebook prepares the cleaned Telecom Churn dataset for ML practices.
*   review the cleaned dataset
*   remove irrelevant columns
*   confirm target variable
*    select final features
*   define x and y
*    split data into training/ testing sets
*   saved the prepared datasets for modelling


# 2.Imports

In [ ]:
# importing packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

# 3.Load Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load Clean Dataset
path = "/content/drive/MyDrive/Thesis/Datasets/telecom_cleaned.csv"

df = pd.read_csv(path)

df.head()

,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,...,State_SD,State_TN,State_TX,State_UT,State_VA,State_VT,State_WA,State_WI,State_WV,State_WY
0,0.692163,-0.527811,-0.335690,1.623917,1.247508,1.579670,0.484868,1.579942,-0.058619,-0.050781,...,False,False,False,False,False,False,False,False,False,False
1,0.161278,-0.527811,-0.335690,1.623917,1.320985,-0.329918,1.135375,-0.330194,-0.095916,0.147654,...,False,False,False,False,False,False,False,False,False,False
2,0.919686,-0.527811,-0.335690,-0.615795,-0.589414,1.179302,0.685024,1.179465,-1.554439,0.494917,...,False,False,False,False,False,False,False,False,False,False
3,-0.420168,-0.692467,2.978938,-0.615795,-0.589414,2.212509,-1.466653,2.212675,-2.718509,-0.596479,...,False,False,False,False,False,False,False,False,False,False
4,-0.647691,-0.527811,2.978938,-0.615795,-0.589414,-0.235822,0.634985,-0.235772,-1.022461,1.090224,...,False,False,False,False,False,False,False,False,False,False


# 4.Confirm Target Variable
The target variable is "Churn", which indicates wether a customer left the service.

- 1 = customer churned
- 0 = customer did not churn

In [ ]:
df["Churn"].value_counts()

,count
Churn,
False,2278
True,388


In [ ]:
df["Churn"].value_counts(normalize=True)*100

,proportion
Churn,
False,85.446362
True,14.553638


# 5.Remove Irrelevant Columns

In [ ]:
df.columns.tolist()

['Account length',
 'Area code',
 'International plan',
 'Voice mail plan',
 'Number vmail messages',
 'Total day minutes',
 'Total day calls',
 'Total day charge',
 'Total eve minutes',
 'Total eve calls',
 'Total eve charge',
 'Total night minutes',
 'Total night calls',
 'Total night charge',
 'Total intl minutes',
 'Total intl calls',
 'Total intl charge',
 'Customer service calls',
 'Churn',
 'State_AL',
 'State_AR',
 'State_AZ',
 'State_CA',
 'State_CO',
 'State_CT',
 'State_DC',
 'State_DE',
 'State_FL',
 'State_GA',
 'State_HI',
 'State_IA',
 'State_ID',
 'State_IL',
 'State_IN',
 'State_KS',
 'State_KY',
 'State_LA',
 'State_MA',
 'State_MD',
 'State_ME',
 'State_MI',
 'State_MN',
 'State_MO',
 'State_MS',
 'State_MT',
 'State_NC',
 'State_ND',
 'State_NE',
 'State_NH',
 'State_NJ',
 'State_NM',
 'State_NV',
 'State_NY',
 'State_OH',
 'State_OK',
 'State_OR',
 'State_PA',
 'State_RI',
 'State_SC',
 'State_SD',
 'State_TN',
 'State_TX',
 'State_UT',
 'State_VA',
 'State_VT',
 '

In [ ]:
cols_to_drop = ['Area code']

In [ ]:
state_cols = [col for col in df.columns if col.startswith("State_")]

In [ ]:
charge_cols = [
    'Total day charge',
    'Total eve charge',
    'Total night charge',
    'Total intl charge'
]

In [ ]:
cols_to_drop = cols_to_drop + state_cols + charge_cols

# Drop all
df = df.drop(columns=cols_to_drop)

print("Remaining Columns:", df.shape[1])

Remaining Columns: 14


## Observations

The following columns were removed:
- area code
- state related variables
- change related variables

These features were removed because they either do not contribute meaningful predictive value, introduce unnecessary complexity or are highly correlated with other features.

# 6.Review Outliers

In [ ]:
df.describe()

,Account length,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total eve minutes,Total eve calls,Total night minutes,Total night calls,Total intl minutes,Total intl calls,Customer service calls
count,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03,2.666000e+03
mean,-1.399231e-16,5.097198e-17,9.594726e-17,-2.665202e-17,1.465861e-16,2.228775e-16,-1.132711e-16,3.198242e-16,5.956725e-16,-1.732381e-17,-2.132161e-16,-5.663553e-17,-7.196044e-17
std,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00,1.000188e+00
min,-2.518430e+00,-3.356901e-01,-6.157949e-01,-5.894135e-01,-3.311458e+00,-5.019422e+00,-3.933617e+00,-4.962065e+00,-3.101565e+00,-3.456440e+00,-3.672045e+00,-1.819157e+00,-1.191955e+00
25%,-6.982511e-01,-3.356901e-01,-6.157949e-01,-5.894135e-01,-6.657103e-01,-6.660292e-01,-6.887477e-01,-6.460883e-01,-6.744811e-01,-6.750593e-01,-6.230740e-01,-5.975267e-01,-4.291724e-01
50%,-1.568400e-02,-3.356901e-01,-6.157949e-01,-5.894135e-01,8.641661e-03,3.451677e-02,1.008679e-02,-1.172304e-03,-3.730931e-04,-5.467553e-03,-1.327980e-02,-1.903165e-01,-4.291724e-01
75%,6.668831e-01,-3.356901e-01,1.623917e+00,8.066471e-01,6.719236e-01,6.850237e-01,6.814391e-01,6.933526e-01,6.954009e-01,6.641242e-01,6.682549e-01,6.241038e-01,3.336100e-01
max,3.599393e+00,2.978938e+00,1.623917e+00,3.084430e+00,3.160845e+00,2.986818e+00,3.205881e+00,3.471452e+00,3.817767e+00,3.393998e+00,3.502005e+00,6.325047e+00,5.673087e+00


## Observations

Outliers were visually identified during the EDA stage using box plots
In this dataset, serveral numerical features contain extreme values. However, these values are likely to represent real customer behavior, rather than errors.

Therefore, outliers are not removed at this stage. Removing them could result in loss of important information relevant to predicting churn. Instead they will be retained and handled implicitly by the chosen ML Model.

Tree based models such as Random Forest and XGBoost are robust to outliers, which supports the decision to retain them.

# 7.Feature Selection
The final feature set is selected by separating the input variables from the target ones.
- x contains the input features used to make predictions
- y contains the churn target variable

In [ ]:
x = df.drop("Churn", axis=1)
y = df["Churn"]

print("X shape:", x.shape)
print("y shape:", y.shape)

X shape: (2666, 13)
y shape: (2666,)


In [ ]:
x.head()

,Account length,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total eve minutes,Total eve calls,Total night minutes,Total night calls,Total intl minutes,Total intl calls,Customer service calls
0,0.692163,-0.335690,1.623917,1.247508,1.579670,0.484868,-0.058619,-0.050781,0.857403,-0.469031,-0.085020,-0.597527,-0.429172
1,0.161278,-0.335690,1.623917,1.320985,-0.329918,1.135375,-0.095916,0.147654,1.048458,0.149054,1.242179,-0.597527,-0.429172
2,0.919686,-0.335690,-0.615795,-0.589414,1.179302,0.685024,-1.554439,0.494917,-0.759668,0.200561,0.704125,0.216894,-1.191955
3,-0.420168,2.978938,-0.615795,-0.589414,2.212509,-1.466653,-2.718509,-0.596479,-0.084083,-0.572045,-1.304609,1.031314,0.333610
4,-0.647691,2.978938,-0.615795,-0.589414,-0.235822,0.634985,-1.022461,1.090224,-0.281046,1.076181,-0.049150,-0.597527,1.096392


In [ ]:
y.head()

,Churn
0,False
1,False
2,False
3,False
4,False


# 8.Train-Test Split
The dataset is split into training and testing sets.

- 80% of the data is used for training the ML
- 20% of the data is used for testing

A fixed random_state is used to make the split reproducible.

In [18]:
# Creating the Split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )

print("X Train Shape:", x_train.shape)
print("y Train Shape:", y_train.shape)
print("X Test Shape:", x_test.shape)
print("y Test Shape:", y_test.shape)

X Train Shape: (2132, 13)
y Train Shape: (2132,)
X Test Shape: (534, 13)
y Test Shape: (534,)


In [19]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True)*100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True)*100)

Training target distribution:
Churn
False    85.459662
True     14.540338
Name: proportion, dtype: float64

Testing target distribution:
Churn
False    85.393258
True     14.606742
Name: proportion, dtype: float64


# Observations
The distribution of the target variable remains consistent between the training and testing datasets.

This indicates that stratified sampling was successfully applied, ensuring that both datasets reflect the original class distribution.

Maintaining this balance is important for reliable model training and evaluation, particularly in the presence of class imbalance.

# 8.1Save Prepared Data

In [22]:
save_folder = "/content/drive/MyDrive/Thesis/Prepared Data/"

x_train.to_csv(save_folder + "x_train.csv", index=False)
x_test.to_csv(save_folder + "x_test.csv", index=False)
y_train.to_csv(save_folder + "y_train.csv", index=False)
y_test.to_csv(save_folder + "y_test.csv", index=False)